# CS521 HW3 — Problem 1

Jinwu Hu (jinwuhu2)

Fresh five-epoch standard and L-infinity adversarial training using the [official Q3 scaffold](https://github.com/nalinwadhwa02/CCS521FA26HW/blob/main/hw3/q3/multi_norm_adversarial_training_cifar10.ipynb). Neither model loads a base checkpoint. This notebook embeds all Python source and downloads only the two public CIFAR-10 subset files. Select a GPU runtime to reproduce training.

In [ ]:
%pip install -q torch==2.9.0 numpy==2.2.6 gdown


In [ ]:
from pathlib import Path
import hashlib,json,subprocess,sys
root=Path("/content/cs521_hw3_q1") if Path("/content").exists() else Path.cwd()/"cs521_hw3_q1"
root.mkdir(parents=True,exist_ok=True)


## Embedded source: q3_multinorm.py

In [ ]:
(root/'q3_multinorm.py').write_text('"""CS521 HW3 Q3: completed scaffold and reproducible experiment runner.\nAdapted from the official nalinwadhwa02/CCS521FA26HW notebook.\n"""\nimport argparse\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nROOT = Path(__file__).resolve().parent.parent\nDATA_DIR = ROOT / "data"\nimport time\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\ndevice = torch.device(os.environ.get("CS521_DEVICE", "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")))\nprint("device:", device)\n\nNORMS = ("linf", "l2", "l1")\nEPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}\n\ndef load_data(path):\n    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.\n    x, y = torch.load(DATA_DIR / path, map_location="cpu", weights_only=True)\n    return x.float().div(255).to(device), y.to(device)\n\n\n# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.\ndef initialize_data():\n    global x_train, y_train, x_test, y_test\n    x_train, y_train = load_data("cifar10_train_10k.pt")\n    x_test, y_test = load_data("cifar10_test_1k.pt")\n    print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape), flush=True)\n\n\ndef augment(x):\n    # Random horizontal flip and random crop (padding 4) for each image.\n    n = x.shape[0]\n    flip = torch.rand(n, device=x.device) < 0.5\n    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)\n    padded = F.pad(x, (4, 4, 4, 4))\n    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)\n    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)\n    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()\n\n\ndef batches(x, y, batch_size, shuffle):\n    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)\n    for k in range(0, len(x), batch_size):\n        j = idx[k:k + batch_size]\n        yield x[j], y[j]\n\n# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].\nclass PreActBlock(nn.Module):\n    def __init__(self, in_planes, planes, stride):\n        super().__init__()\n        self.bn1 = nn.BatchNorm2d(in_planes)\n        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)\n        self.bn2 = nn.BatchNorm2d(planes)\n        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)\n        self.shortcut = None\n        if stride != 1 or in_planes != planes:\n            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)\n\n    def forward(self, x):\n        out = F.relu(self.bn1(x))\n        shortcut = self.shortcut(out) if self.shortcut is not None else x\n        out = self.conv1(out)\n        out = self.conv2(F.relu(self.bn2(out)))\n        return out + shortcut\n\n\nclass PreActResNet18(nn.Module):\n    def __init__(self, width=32, num_classes=10):\n        super().__init__()\n        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))\n        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))\n        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)\n        layers, in_planes = [], width\n        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:\n            layers.append(PreActBlock(in_planes, mult * width, stride))\n            in_planes = mult * width\n        self.layers = nn.Sequential(*layers)\n        self.bn = nn.BatchNorm2d(in_planes)\n        self.linear = nn.Linear(in_planes, num_classes)\n\n    def forward(self, x):\n        out = self.layers(self.conv1((x - self.mean) / self.std))\n        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)\n        return self.linear(out)\n\n\ndef make_model():\n    return PreActResNet18().to(device)\n\n\ndef load_base_model():\n    model = make_model()\n    model.load_state_dict(torch.load(DATA_DIR / "linf_base.pt", map_location=device, weights_only=True))\n    return model\n\ndef clip_box(x, delta):\n    # Make x + delta an image in [0, 1]. This never increases |delta_i|.\n    return (x + delta).clamp(0, 1) - x\n\n\ndef input_grad(model, x, delta, y):\n    delta = delta.detach().requires_grad_(True)\n    loss = F.cross_entropy(model(x + delta), y, reduction="sum")\n    (grad,) = torch.autograd.grad(loss, delta)\n    return grad.detach()\n\n\ndef l2_norm(v):\n    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)\n\n\ndef pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):\n    delta = torch.empty_like(x).uniform_(-eps, eps) if rand_init else torch.zeros_like(x)\n    delta = clip_box(x, delta)\n    for _ in range(steps):\n        grad = input_grad(model, x, delta, y)\n        delta = clip_box(x, (delta + alpha * grad.sign()).clamp(-eps, eps))\n    return (x + delta).detach()\n\ndef pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):\n    delta = torch.zeros_like(x)\n    if rand_init:\n        delta = torch.randn_like(x)\n        # Uniform radius, matching the random-radius convention of the supplied L1 attack.\n        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)\n        delta = delta / l2_norm(delta).clamp_min(1e-12) * (eps * r)\n    delta = clip_box(x, delta)\n    for _ in range(steps):\n        grad = input_grad(model, x, delta, y)\n        delta = delta + alpha * grad / l2_norm(grad).clamp_min(1e-12)\n        delta = delta * (eps / l2_norm(delta).clamp_min(1e-12)).clamp(max=1.0)\n        delta = clip_box(x, delta)\n    return (x + delta).detach()\n\n\ndef project_l1(delta, eps):\n    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).\n    v = delta.flatten(1)\n    a = v.abs()\n    outside = a.sum(dim=1) > eps\n    if not outside.any():\n        return delta\n    mu, _ = a.sort(dim=1, descending=True)\n    cumsum = mu.cumsum(dim=1)\n    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)\n    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)\n    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)\n    w = (a - theta).clamp_min(0) * v.sign()\n    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)\n\n\ndef pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):\n    # Each step changes only the k coordinates with the largest gradient.\n    # Coordinates at the image boundary that point outward are skipped.\n    delta = torch.zeros_like(x)\n    if rand_init:\n        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)\n        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)\n        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r\n    delta = clip_box(x, delta)\n    for _ in range(steps):\n        grad = input_grad(model, x, delta, y).flatten(1)\n        x_cur = (x + delta).flatten(1)\n        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))\n        grad = grad.masked_fill(blocked, 0)\n        top = grad.abs().topk(k, dim=1).indices\n        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())\n        delta = clip_box(x, delta + alpha * (step / k).view_as(x))\n        delta = project_l1(delta, eps)\n    return (x + delta).detach()\n\n\nATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}\n\n\ndef attack(model, x, y, norm, steps=10):\n    # The attack for one norm with the default eps and step size.\n    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps\n    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)\n\ndef test_model(model, steps=50, restarts=2, batch_size=250):\n    model.eval()\n    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable\n    start = time.time()\n    correct = {k: 0 for k in ("clean", *NORMS, "union")}\n    for i in range(0, len(x_test), batch_size):\n        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]\n        with torch.no_grad():\n            clean = model(x).argmax(1) == y\n        union = clean.clone()\n        for norm in NORMS:\n            robust = clean.clone()\n            for _ in range(restarts):\n                x_adv = attack(model, x, y, norm, steps)\n                with torch.no_grad():\n                    robust &= model(x_adv).argmax(1) == y\n            correct[norm] += robust.sum().item()\n            union &= robust\n        correct["clean"] += clean.sum().item()\n        correct["union"] += union.sum().item()\n    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}\n    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")\n    return acc\n\ndef adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):\n    torch.manual_seed(seed)\n    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)\n    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))\n    params = list(model.parameters())\n    start = time.time()\n    history = []\n    for epoch in range(epochs):\n        sync_device()\n        epoch_start = time.perf_counter()\n        losses = []\n        for x, y in batches(x_train, y_train, batch_size, shuffle=True):\n            if len(y) < batch_size:  # skip the last, smaller batch\n                continue\n            x = augment(x)\n            loss = loss_fn(model, x, y, **kwargs)\n            opt.zero_grad()\n            if gp_beta is None:\n                loss.backward()\n            else:\n                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n\n                # for each parameter tensor. gradient_projection is defined in part (b).\n                g_a = torch.autograd.grad(loss, params)\n                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)\n                for p, ga, gn in zip(params, g_a, g_n):\n                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga\n            opt.step()\n            sched.step()\n            losses.append(loss.item())\n        sync_device()\n        history.append({"epoch": epoch + 1, "loss": float(np.mean(losses)), "seconds": time.perf_counter() - epoch_start})\n        print(json.dumps(history[-1]), flush=True)\n    model.eval()\n    model.training_history = history\n    return model\n\ndef single_norm_loss(model, x, y, norm):\n    model.eval()\n    x_adv = attack(model, x, y, norm)\n    model.train()\n    return F.cross_entropy(model(x_adv), y)\n\n\ndef ramp_loss(model, x, y, lam=0.5):\n    model.eval()\n    x_q = attack(model, x, y, "linf")\n    x_r = attack(model, x, y, "l1")\n    model.train()\n    logits_q, logits_r = model(x_q), model(x_r)\n    ce_q = F.cross_entropy(logits_q, y, reduction="none")\n    ce_r = F.cross_entropy(logits_r, y, reduction="none")\n    # PyTorch\'s target is p_q, input is log p_r: this is KL(p_q || p_r).\n    kl = F.kl_div(F.log_softmax(logits_r, dim=1), F.softmax(logits_q, dim=1),\n                  reduction="none").sum(dim=1)\n    mask = (logits_q.argmax(1) == y).to(kl.dtype)\n    # Mean over the qualifying examples; exactly zero when none qualify.\n    paired = (kl * mask).sum() / mask.sum().clamp_min(1)\n    return torch.maximum(ce_q, ce_r).mean() + lam * paired\n\ndef gradient_projection(g_n, g_a):\n    cosine = (g_n.flatten() @ g_a.flatten()) / (g_n.norm() * g_a.norm()).clamp_min(1e-12)\n    return cosine.clamp_min(0) * g_n\n\n\nclass AlternatingDistillationLoss:\n    """Alternate L1/L2 updates while retaining an L-infinity-robust teacher.\n\n    This is a student-method candidate, not a claim of global research novelty.\n    Teacher targets never depend on test examples or test labels.\n    """\n    def __init__(self, teacher, distill_weight=0.5, clean_weight=0.25):\n        self.teacher = teacher.eval()\n        for p in self.teacher.parameters():\n            p.requires_grad_(False)\n        self.distill_weight = distill_weight\n        self.clean_weight = clean_weight\n        self.batch_index = 0\n        self.stats = []\n\n    def __call__(self, model, x, y):\n        norm = ("l1", "l2")[self.batch_index % 2]\n        self.batch_index += 1\n        model.eval()\n        x_alt = attack(model, x, y, norm)\n        # Keep attack cost identical in the ablation to isolate distillation.\n        x_anchor = attack(model, x, y, "linf")\n        with torch.no_grad():\n            teacher_logits = self.teacher(x)\n            teacher_prob = F.softmax(teacher_logits, dim=1)\n            reliable = (teacher_logits.argmax(1) == y)\n        model.train()\n        alt_logits = model(x_alt)\n        anchor_logits = model(x_anchor)\n        clean_logits = model(x)\n        ce_alt = F.cross_entropy(alt_logits, y)\n        ce_clean = F.cross_entropy(clean_logits, y)\n        kd_each = F.kl_div(F.log_softmax(anchor_logits, dim=1), teacher_prob,\n                          reduction="none").sum(dim=1)\n        mask = reliable.to(kd_each.dtype)\n        kd = (kd_each * mask).sum() / mask.sum().clamp_min(1)\n        loss = ce_alt + self.distill_weight * kd + self.clean_weight * ce_clean\n        self.stats.append({"norm": norm, "ce_alt": ce_alt.detach().item(),\n                           "clean_ce": ce_clean.detach().item(), "distillation": kd.detach().item(),\n                           "teacher_correct_fraction": mask.mean().item()})\n        return loss\n\n_my_loss_instance = None\ndef my_loss(model, x, y):\n    global _my_loss_instance\n    if _my_loss_instance is None:\n        _my_loss_instance = AlternatingDistillationLoss(load_base_model())\n    return _my_loss_instance(model, x, y)\n\n\n\ndef sync_device():\n    if device.type == "cuda": torch.cuda.synchronize()\n    elif device.type == "mps": torch.mps.synchronize()\n\ndef save_model(model, path):\n    torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, path)\n\ndef execute_run(method, seed, epochs=10):\n    dest = ROOT / "results/q3" / f"{method}_seed{seed}_epochs{epochs}"\n    dest.mkdir(parents=True, exist_ok=True)\n    result_path = dest / "result.json"\n    if result_path.exists():\n        print("SKIP_COMPLETE", dest.name, flush=True)\n        return json.loads(result_path.read_text())\n    torch.manual_seed(seed)\n    model = load_base_model()\n    loss_fn = None\n    extra = {}\n    if method in NORMS:\n        loss_fn = single_norm_loss\n        extra["norm"] = method\n    elif method == "ramp":\n        loss_fn = ramp_loss\n        extra["gp_beta"] = 0.5\n    elif method in ["alternating_distillation", "without_distillation"]:\n        loss_fn = AlternatingDistillationLoss(load_base_model(),\n                    distill_weight=0.5 if method == "alternating_distillation" else 0.0)\n    elif method != "base":\n        raise ValueError(method)\n    if loss_fn is not None:\n        model = adv_train(model, loss_fn, epochs=epochs, seed=seed, **extra)\n        save_model(model, dest / "model.pt")\n        (dest / "training_history.json").write_text(json.dumps(model.training_history, indent=2) + "\\n")\n        if isinstance(loss_fn, AlternatingDistillationLoss):\n            (dest / "batch_statistics.json").write_text(json.dumps(loss_fn.stats) + "\\n")\n    sync_device()\n    start = time.perf_counter()\n    acc = test_model(model)  # Required 1,000 examples, 50 steps, two restarts.\n    sync_device()\n    result = {"method": method, "seed": seed, "epochs": epochs if method != "base" else 0,\n              "accuracy_percent": acc, "evaluation_seconds": time.perf_counter() - start,\n              "training_history": getattr(model, "training_history", []),\n              "evaluation": {"n": 1000, "steps": 50, "restarts": 2, "batch_size": 250},\n              "device": str(device), "torch": torch.__version__,\n              "source_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest()}\n    result_path.write_text(json.dumps(result, indent=2) + "\\n")\n    print("RUN_COMPLETE", method, seed, json.dumps(acc), flush=True)\n    del model, loss_fn\n    if device.type == "cuda": torch.cuda.empty_cache()\n    elif device.type == "mps": torch.mps.empty_cache()\n    return result\n\ndef benchmark():\n    model = load_base_model()\n    model.eval()\n    x, y = x_train[:128], y_train[:128]\n    for norm in NORMS:\n        sync_device()\n        start = time.perf_counter()\n        x_adv = attack(model, x, y, norm, steps=10)\n        sync_device()\n        print("BENCHMARK", norm, time.perf_counter() - start, flush=True)\n        delta = x_adv - x\n        ord_value = {"linf": float("inf"), "l2": 2, "l1": 1}[norm]\n        max_norm = delta.flatten(1).norm(p=ord_value, dim=1).max().item()\n        assert max_norm <= EPS[norm] + 1e-4, (norm, max_norm)\n        assert x_adv.min() >= 0 and x_adv.max() <= 1\n        assert not model.training\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--method", default="benchmark", choices=["benchmark","base",*NORMS,"ramp","alternating_distillation","without_distillation","all","strong"])\n    p.add_argument("--seed", type=int, default=0)\n    p.add_argument("--epochs", type=int, default=10)\n    p.add_argument("--checkpoint")\n    a = p.parse_args()\n    torch.set_num_threads(4)\n    initialize_data()\n    if a.method == "benchmark":\n        benchmark()\n    elif a.method == "all":\n        schedule = [("base",0),("linf",0),("l2",0),("l1",0),("ramp",0),("ramp",1),\n                    ("alternating_distillation",0),("alternating_distillation",1),\n                    ("without_distillation",0),("without_distillation",1)]\n        for method, seed in schedule: execute_run(method, seed)\n    elif a.method == "strong":\n        if not a.checkpoint: raise ValueError("--checkpoint is required")\n        model = make_model()\n        model.load_state_dict(torch.load(a.checkpoint, map_location=device, weights_only=True))\n        acc = test_model(model, steps=200, restarts=5)\n        Path(a.checkpoint).with_name("strong_evaluation.json").write_text(\n            json.dumps({"accuracy_percent": acc, "steps":200, "restarts":5, "n":1000}, indent=2) + "\\n")\n    else:\n        execute_run(a.method, a.seed, a.epochs)\n\nif __name__ == "__main__":\n    main()\n')


## Embedded source: run_q1.py

In [ ]:
(root/'run_q1.py').write_text('"""Fresh five-epoch Q1 experiment with resumable per-epoch checkpoints."""\nimport argparse\nimport hashlib\nimport json\nimport platform\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nimport q3_multinorm as s\n\ndef now():return datetime.now(timezone.utc).isoformat()\ndef digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()\ndef save_json(path,value):\n    tmp=path.with_suffix(\'.json.tmp\');tmp.write_text(json.dumps(value,indent=2)+\'\\n\');tmp.replace(path)\ndef natural_loss(model,x,y):\n    model.train();return F.cross_entropy(model(x),y)\ndef adversarial_loss(model,x,y):\n    model.eval();adv=s.pgd_linf(model,x,y,8/255,2/255,10,True)\n    model.train();return F.cross_entropy(model(adv),y)\n\ndef train(name,dest):\n    torch.manual_seed(0)\n    model=s.make_model()\n    opt=torch.optim.SGD(model.parameters(),lr=.01,momentum=.9,weight_decay=5e-4)\n    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,5*(len(s.y_train)//128))\n    torch.manual_seed(0)  # Match the official adv_train loop\'s RNG reset.\n    checkpoint=dest/\'training_checkpoint.pt\';history=[]\n    gpu_api=getattr(torch,s.device.type,None)\n    if checkpoint.exists():\n        state=torch.load(checkpoint,map_location=\'cpu\',weights_only=True)\n        model.load_state_dict(state[\'model\']);opt.load_state_dict(state[\'optimizer\']);sched.load_state_dict(state[\'scheduler\'])\n        torch.set_rng_state(state[\'cpu_rng\'])\n        if state.get(\'gpu_rng\') is not None:gpu_api.set_rng_state(state[\'gpu_rng\'])\n        history=state[\'history\'];print(\'RESUME\',name,len(history),flush=True)\n    loss_fn=natural_loss if name==\'standard\' else adversarial_loss\n    for epoch in range(len(history),5):\n        s.sync_device();started=time.perf_counter();losses=[]\n        for x,y in s.batches(s.x_train,s.y_train,128,shuffle=True):\n            if len(y)<128:continue\n            x=s.augment(x);loss=loss_fn(model,x,y)\n            opt.zero_grad();loss.backward();opt.step();sched.step();losses.append(loss.item())\n        s.sync_device();history.append(dict(epoch=epoch+1,loss=float(np.mean(losses)),seconds=time.perf_counter()-started,completed_at=now()))\n        state=dict(model={k:v.detach().cpu() for k,v in model.state_dict().items()},optimizer=opt.state_dict(),scheduler=sched.state_dict(),history=history,cpu_rng=torch.get_rng_state(),gpu_rng=gpu_api.get_rng_state() if s.device.type in [\'cuda\',\'mps\'] else None)\n        tmp=dest/\'training_checkpoint.pt.tmp\';torch.save(state,tmp);tmp.replace(checkpoint)\n        save_json(dest/\'training_history.json\',history)\n        print(\'EPOCH\',name,json.dumps(history[-1]),flush=True)\n    s.save_model(model,dest/\'model.pt\');return model.eval()\n\ndef evaluate(model,dest):\n    out=dest/\'evaluation.json\';records=json.loads(out.read_text()) if out.exists() else []\n    for radius in [0,2,4,8,12]:\n        if any(r[\'epsilon_over_255\']==radius for r in records):continue\n        torch.manual_seed(42);counts=dict(clean=0,fgsm=0,pgd=0);per_example=[]\n        s.sync_device();started=time.perf_counter()\n        for start in range(0,len(s.y_test),100):\n            x,y=s.x_test[start:start+100],s.y_test[start:start+100]\n            with torch.no_grad():clean=model(x).argmax(1).eq(y)\n            if radius==0:fgsm,pgd=clean.clone(),clean.clone()\n            else:\n                eps=radius/255\n                fx=s.pgd_linf(model,x,y,eps,eps,1,False)\n                with torch.no_grad():fgsm=clean & model(fx).argmax(1).eq(y)\n                pgd=fgsm.clone()\n                for _ in range(2):\n                    ax=s.pgd_linf(model,x,y,eps,eps/4,50,True)\n                    with torch.no_grad():pgd &= model(ax).argmax(1).eq(y)\n            masks={k:v.cpu().tolist() for k,v in dict(clean=clean,fgsm=fgsm,pgd=pgd).items()}\n            for k,v in masks.items():counts[k]+=sum(v)\n            per_example.extend(dict(test_index=start+i,**{k:bool(v[i]) for k,v in masks.items()}) for i in range(len(y)))\n        s.sync_device();row=dict(epsilon_over_255=radius,counts=counts,n_test=len(s.y_test),accuracy_percent={k:100*v/len(s.y_test) for k,v in counts.items()},elapsed_seconds=time.perf_counter()-started,per_example=per_example)\n        records.append(row);save_json(out,records);print(\'EVALUATION\',dest.name,radius,row[\'accuracy_percent\'],flush=True)\n\ndef main():\n    p=argparse.ArgumentParser();p.add_argument(\'--data-dir\',type=Path,required=True);p.add_argument(\'--output\',type=Path,default=Path(\'results\'));a=p.parse_args()\n    torch.set_num_threads(4);s.DATA_DIR=a.data_dir;s.initialize_data()\n    assert len(s.y_train)==10000 and len(s.y_test)==1000\n    a.output.mkdir(exist_ok=True,parents=True)\n    protocol=dict(started_at=now(),epochs=5,seed=0,architecture=\'Official Q3 width-32 PreActResNet18 with ReLU and input normalization\',initialization=\'Each model starts from seed 0 and make_model; no base or pretrained model is loaded. Interrupted runs resume only their own epoch checkpoint.\',optimizer=\'SGD\',learning_rate=.01,momentum=.9,weight_decay=5e-4,batch_size=128,drop_last=True,scheduler=\'CosineAnnealingLR over 5 * floor(10000/128) steps\',augmentation=\'Official random crop padding 4 and horizontal flip\',train_size=10000,test_size=1000,training_attack=dict(norm=\'linf\',epsilon=8/255,steps=10,step_size=2/255,random_start=True),evaluation=dict(radii_over_255=[0,2,4,8,12],pgd_steps=50,restarts=2,step_size=\'epsilon/4\',robust_rule=\'clean-correct and correct on FGSM plus both PGD endpoint candidates\'),device=str(s.device),torch=torch.__version__,numpy=np.__version__,platform=platform.platform(),data_sha256={n:digest(a.data_dir/n) for n in [\'cifar10_train_10k.pt\',\'cifar10_test_1k.pt\']},source_sha256={n:digest(Path(__file__).parent/n) for n in [\'run_q1.py\',\'q3_multinorm.py\']})\n    f=a.output/\'protocol.json\'\n    if f.exists():assert json.loads(f.read_text())[\'source_sha256\']==protocol[\'source_sha256\']\n    else:save_json(f,protocol)\n    for name in [\'standard\',\'linf\']:\n        d=a.output/name;d.mkdir(exist_ok=True)\n        model=train(name,d);evaluate(model,d);del model\n        if s.device.type==\'mps\':torch.mps.empty_cache()\n    save_json(a.output/\'completed.json\',dict(completed_at=now()));print(\'COMPLETE\',flush=True)\nif __name__==\'__main__\':main()\n')


In [ ]:
import gdown
data=root/"data"
data.mkdir(exist_ok=True)
assets={"cifar10_train_10k.pt":("1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe","dac9de04f3bfc3fea36381fe62894b34f7b66f73de46f3ac2502e567f3816c00"),"cifar10_test_1k.pt":("1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA","cb14f0bede10d8dd0ee36c1bc58f5d11e69d61ff9834918a6318b8f63411ce7a")}
for name,(file_id,expected) in assets.items():
    if not (data/name).exists():gdown.download(id=file_id,output=str(data/name),quiet=False)
    assert hashlib.sha256((data/name).read_bytes()).hexdigest()==expected,name


## Training and evaluation

Both models use seed 0, five epochs, batch size 128, SGD with learning rate 0.01, momentum 0.9, weight decay 0.0005 and cosine scheduling. The adversarial model uses ten-step random-start PGD at epsilon 8/255 and step size 2/255. The first 1,000 test images are evaluated at radii 0/2/4/8/12 over 255 using FGSM and 50-step PGD with two restarts. The PGD candidate pool includes FGSM; all robust counts require clean correctness. Per-epoch checkpoints support resuming the same run. Use a new output directory for an independent fresh run.

In [ ]:
RUN_TRAINING=False
if RUN_TRAINING:
    subprocess.run([sys.executable,str(root/"run_q1.py"),"--data-dir",str(data),"--output",str(root/"fresh_results")],cwd=root,check=True)
else:
    print("Saved evidence only. Set RUN_TRAINING=True to reproduce the experiment.")


## Recorded experiment

This literal record comes from the completed local Apple MPS run with PyTorch 2.9.0. It is not a claim that this new notebook session has run training. The archive includes all per-example outcomes and source hashes. The earlier warm-start experiment is retained separately as historical evidence. CUDA and MPS may differ numerically; five epochs and one seed limit conclusions. Finite-step attack results are empirical upper bounds on worst-case robustness, not certificates.

In [ ]:
recorded_results={'standard': {'training_history': [{'epoch': 1, 'loss': 1.8899560601283343, 'seconds': 7.530434334000347, 'completed_at': '2026-10-24T17:03:16.493080+00:00'}, {'epoch': 2, 'loss': 1.5429040529789069, 'seconds': 6.432032582999909, 'completed_at': '2026-10-24T17:03:23.025067+00:00'}, {'epoch': 3, 'loss': 1.3493580252696307, 'seconds': 6.4260478330002115, 'completed_at': '2026-10-24T17:03:29.537182+00:00'}, {'epoch': 4, 'loss': 1.216496921502627, 'seconds': 6.286863833000098, 'completed_at': '2026-10-24T17:03:35.918115+00:00'}, {'epoch': 5, 'loss': 1.1303420899770198, 'seconds': 6.350155083000118, 'completed_at': '2026-10-24T17:03:42.366571+00:00'}], 'evaluation': [{'epsilon_over_255': 0, 'counts': {'clean': 579, 'fgsm': 579, 'pgd': 579}, 'n_test': 1000, 'accuracy_percent': {'clean': 57.9, 'fgsm': 57.9, 'pgd': 57.9}, 'elapsed_seconds': 0.2069118750000598}, {'epsilon_over_255': 2, 'counts': {'clean': 579, 'fgsm': 266, 'pgd': 208}, 'n_test': 1000, 'accuracy_percent': {'clean': 57.9, 'fgsm': 26.6, 'pgd': 20.8}, 'elapsed_seconds': 40.287308875000235}, {'epsilon_over_255': 4, 'counts': {'clean': 579, 'fgsm': 116, 'pgd': 57}, 'n_test': 1000, 'accuracy_percent': {'clean': 57.9, 'fgsm': 11.6, 'pgd': 5.7}, 'elapsed_seconds': 39.93646933299988}, {'epsilon_over_255': 8, 'counts': {'clean': 579, 'fgsm': 32, 'pgd': 3}, 'n_test': 1000, 'accuracy_percent': {'clean': 57.9, 'fgsm': 3.2, 'pgd': 0.3}, 'elapsed_seconds': 80.4478227919999}, {'epsilon_over_255': 12, 'counts': {'clean': 579, 'fgsm': 13, 'pgd': 0}, 'n_test': 1000, 'accuracy_percent': {'clean': 57.9, 'fgsm': 1.3, 'pgd': 0.0}, 'elapsed_seconds': 129.14880483399975}]}, 'linf': {'training_history': [{'epoch': 1, 'loss': 2.1814850721603785, 'seconds': 133.3537826249999, 'completed_at': '2026-10-24T17:10:46.152284+00:00'}, {'epoch': 2, 'loss': 2.1000344034952994, 'seconds': 127.82683454199969, 'completed_at': '2026-10-24T17:12:54.192637+00:00'}, {'epoch': 3, 'loss': 2.0522137712209654, 'seconds': 133.76246425, 'completed_at': '2026-10-24T17:15:08.119780+00:00'}, {'epoch': 4, 'loss': 2.02878478093025, 'seconds': 366.65236483299987, 'completed_at': '2026-10-24T17:21:14.967722+00:00'}, {'epoch': 5, 'loss': 2.017888021774781, 'seconds': 434.34878445799995, 'completed_at': '2026-10-24T17:28:29.788256+00:00'}], 'evaluation': [{'epsilon_over_255': 0, 'counts': {'clean': 385, 'fgsm': 385, 'pgd': 385}, 'n_test': 1000, 'accuracy_percent': {'clean': 38.5, 'fgsm': 38.5, 'pgd': 38.5}, 'elapsed_seconds': 1.7102661250000892}, {'epsilon_over_255': 2, 'counts': {'clean': 385, 'fgsm': 344, 'pgd': 343}, 'n_test': 1000, 'accuracy_percent': {'clean': 38.5, 'fgsm': 34.4, 'pgd': 34.3}, 'elapsed_seconds': 297.19455554199976}, {'epsilon_over_255': 4, 'counts': {'clean': 385, 'fgsm': 299, 'pgd': 298}, 'n_test': 1000, 'accuracy_percent': {'clean': 38.5, 'fgsm': 29.9, 'pgd': 29.8}, 'elapsed_seconds': 387.9768946659997}, {'epsilon_over_255': 8, 'counts': {'clean': 385, 'fgsm': 245, 'pgd': 232}, 'n_test': 1000, 'accuracy_percent': {'clean': 38.5, 'fgsm': 24.5, 'pgd': 23.2}, 'elapsed_seconds': 601.3823484579998}, {'epsilon_over_255': 12, 'counts': {'clean': 385, 'fgsm': 194, 'pgd': 172}, 'n_test': 1000, 'accuracy_percent': {'clean': 38.5, 'fgsm': 19.4, 'pgd': 17.2}, 'elapsed_seconds': 487.5439670420001}]}}
print(json.dumps(recorded_results,indent=2))
